This framework was built to simulate how revenue drivers, retention dynamics, and capital allocation decisions translate into financial performance, cash sustainability, and enterprise value. It is designed to mirror how a Corporate FP&A team supports executive decision-making in a high-growth SaaS environment.

In [ ]:
import pandas as pd
import numpy as np

# Load dataset
df = pd.read_csv("/content/Telco-Customer-Churn.csv")

# Clean TotalCharges
df["TotalCharges"] = df["TotalCharges"].replace(" ", pd.NA)
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"])
df["TotalCharges"] = df["TotalCharges"].fillna(0)

# Create churn flag
df["churn_flag"] = df["Churn"].map({"Yes": 1, "No": 0})

In [ ]:
# Assign plan tier based on MonthlyCharges
def assign_plan(x):
    if x < 40:
        return "Basic"
    elif x < 90:
        return "Pro"
    else:
        return "Enterprise"

df["plan_tier"] = df["MonthlyCharges"].apply(assign_plan)

# Simulated pricing model
pricing_map = {
    "Basic": 30,
    "Pro": 80,
    "Enterprise": 250
}

df["simulated_mrr"] = df["plan_tier"].map(pricing_map)

# Scale to $35M ARR
starting_arr = 35000000
raw_arr = df["simulated_mrr"].sum() * 12
scaling_factor = starting_arr / raw_arr

df["scaled_mrr"] = df["simulated_mrr"] * scaling_factor

starting_mrr = df["scaled_mrr"].sum()
starting_arr = starting_mrr * 12

starting_mrr, starting_arr

(np.float64(2916666.6666666665), np.float64(35000000.0))

In [ ]:
# Logo churn
logo_churn_rate = df["churn_flag"].mean()

# Revenue churn
churn_revenue = df.loc[df["churn_flag"] == 1, "scaled_mrr"].sum()
total_revenue = df["scaled_mrr"].sum()

revenue_churn_rate = churn_revenue / total_revenue
grr = 1 - revenue_churn_rate

# Expansion assumption (15%)
expansion_rate = 0.15

retained_revenue = total_revenue - churn_revenue
expanded_revenue = retained_revenue * (1 + expansion_rate)

nrr = expanded_revenue / total_revenue

logo_churn_rate, grr, nrr

(np.float64(0.2653698707936959),
 np.float64(0.6923717890296458),
 np.float64(0.7962275573840925))

In [ ]:
# Create cohort month
max_tenure = df["tenure"].max()
df["cohort_month"] = max_tenure - df["tenure"]

# Active customers
df["active"] = 1 - df["churn_flag"]

cohort_counts = df.groupby(["cohort_month", "tenure"])["active"].sum().reset_index()

cohort_pivot = cohort_counts.pivot(index="cohort_month", columns="tenure", values="active")

cohort_sizes = cohort_pivot.max(axis=1)

retention_matrix = cohort_pivot.divide(cohort_sizes, axis=0)

retention_matrix.head()

tenure,0,1,2,3,4,5,6,7,8,9,...,63,64,65,66,67,68,69,70,71,72
cohort_month,,,,,,,,,,,,,,,,,,,,,
0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0
1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0,NaN
2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0,NaN,NaN
3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,1.0,NaN,NaN,NaN
4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,1.0,NaN,NaN,NaN,NaN


In [ ]:
monthly_churn = logo_churn_rate / 12
months = np.arange(0, 37)

retention_curve = (1 - monthly_churn) ** months

blended_arpu = 80
gross_margin = 0.75
monthly_margin = blended_arpu * gross_margin

ltv = (retention_curve * monthly_margin).sum()

ltv

np.float64(1527.0355772955506)

In [ ]:
plan_ltv = {}

for plan in pricing_map:

    churn = df[df["plan_tier"] == plan]["churn_flag"].mean() / 12
    retention = (1 - churn) ** months

    arpu = pricing_map[plan]
    margin = arpu * gross_margin

    plan_ltv[plan] = (retention * margin).sum()

plan_ltv

{'Basic': np.float64(702.7705574188695),
 'Pro': np.float64(1435.486182137347),
 'Enterprise': np.float64(4396.399161257906)}

In [ ]:
monthly_churn = 0.02
monthly_expansion = 0.02
new_business_growth = 0.03

months = 12
current_mrr = starting_mrr

for _ in range(months):
    net_existing = current_mrr * (1 - monthly_churn + monthly_expansion)
    new_gain = current_mrr * new_business_growth
    current_mrr = net_existing + new_gain

final_arr = current_mrr * 12
growth_rate = (final_arr - starting_arr) / starting_arr

final_arr, growth_rate

(np.float64(49901631.03961626), np.float64(0.42576088684617874))

In [ ]:
simulations = 1000
results = []

for _ in range(simulations):

    current_mrr = starting_mrr

    for _ in range(12):
        churn = np.random.uniform(0.015, 0.03)
        expansion = np.random.uniform(0.015, 0.025)
        acquisition = np.random.uniform(0.025, 0.035)

        net_existing = current_mrr * (1 - churn + expansion)
        new_gain = current_mrr * acquisition

        current_mrr = net_existing + new_gain

    results.append(current_mrr * 12)

results = np.array(results)

mean_projection = results.mean()
probability_below_target = np.mean(results < 49000000)

mean_projection, probability_below_target

(np.float64(48450669.51489917), np.float64(0.704))

This model functions as a decision-support system, enabling leadership to evaluate growth efficiency, capital allocation tradeoffs, and long-term value creation under multiple operating scenarios.